# 2.8 — Structured products and correlation foundations

This lab builds the two-asset cash pool and one-period waterfall introduced in lesson 2.8, then measures the first-writedown threshold on the same base-stage structured market. The later C3 credit extension is intentionally absent.


## Structured-note redemption: known fixings before pricing uncertainty

A note is a debt claim with an embedded payoff. Use the supported JSON instrument route to evaluate three final-redemption contracts after all observation fixings are known. Coupons are zero, the autocall threshold is deliberately unreachable in this grid, and the discount rate is zero; those controls isolate principal repayment from coupons, discounting, and simulation noise. This is a contract test, not a quote for a newly issued note or an issuer-credit model.

The capital-protection contract uses a floor and a cap on the total spot ratio. The participation contract returns principal plus capped upside. The knock-in contract returns principal less the embedded put loss after a discrete barrier breach. A breach followed by full recovery returns principal.


In [ ]:
import datetime as dt
import json
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.core.money import Money
from finstack_quant.valuations.instruments import price_instrument

note_market = MarketContext().insert(DiscountCurve.flat("USD-OIS", dt.date(2026, 1, 1), 0.0))
note_market.insert_price("SPX-SPOT", Money(100.0, "USD"))
note_spec = {
    "id": "NOTE-PAYOFF-PROOF", "underlying_ticker": "SPX",
    "observation_dates": ["2025-12-30", "2025-12-31"],
    "payment_dates": ["2026-01-02", "2026-01-02"], "expiry": "2026-01-02",
    "autocall_barriers": [3.0, 3.0], "coupon_barriers": [0.0, 0.0],
    "coupons": [0.0, 0.0], "memory_coupons": False,
    "final_barrier": 0.7, "participation_rate": 1.0, "cap_level": 1.5,
    "notional": {"amount": "1000", "currency": "USD"}, "day_count": "act_365f",
    "discount_curve_id": "USD-OIS", "spot_id": "SPX-SPOT", "vol_surface_id": "SPX-VOL",
    "path_model": "atm_term_gbm", "div_yield_id": None, "initial_level": 100.0,
    "attributes": {},
}
print("terminal | capital floor/cap | 50% upside | knock-in principal")
for terminal in (50.0, 70.0, 90.0, 100.0, 125.0, 160.0):
    note_spec["past_fixings"] = [["2025-12-30", 100.0], ["2025-12-31", terminal]]
    ratio = terminal / 100.0
    contracts = [
        ({"capital_protection": {"floor": 1.0}}, max(1.0, min(ratio, 1.5))),
        ({"participation": {"rate": 0.5}}, 1.0 + 0.5 * max(min(ratio, 1.5) - 1.0, 0.0)),
        ({"knock_in_put": {"strike": 100.0}}, ratio if ratio <= 0.7 else 1.0),
    ]
    values = []
    for payoff, expected in contracts:
        note_spec["final_payoff_type"] = payoff
        envelope = {"schema": "finstack_quant.instrument/1",
                    "instrument": {"type": "autocallable", "spec": note_spec}}
        value = price_instrument(json.dumps(envelope), note_market, "2026-01-01").price
        assert abs(value - 1000 * expected) < 1e-10
        values.append(value)
    print(f"{terminal:8.0f} | {values[0]:17.2f} | {values[1]:10.2f} | {values[2]:18.2f}")
note_spec["past_fixings"] = [["2025-12-30", 50.0], ["2025-12-31", 100.0]]
envelope["instrument"]["spec"] = note_spec
recovered = price_instrument(json.dumps(envelope), note_market, "2026-01-01").price
assert recovered == 1000.0
print(f"Prior knock-in with full recovery: {recovered:.2f}")


## Read the coupon promise as arithmetic

A quoted coupon rate is not a bond yield. A TARN stops at a cumulative target, a snowball carries the prior coupon into the next reset, and a range accrual earns only the observed fraction within its range. These deterministic helpers expose coupon mechanics; they do not value issuer credit, future rate uncertainty, or a call right. In particular, `callable_range_accrual_accrued` computes the uncalled period's coupon and does not price the callable instrument.


In [ ]:
from finstack_quant.valuations import (
    tarn_coupon_profile, snowball_coupon_profile,
    inverse_floater_coupon_profile, callable_range_accrual_accrued,
)

coupon_tarn = tarn_coupon_profile(0.05, 0.0, [0.02, 0.03, 0.04], 0.025, 0.5)
# First payment is (5%-2%)*0.5=1.5%; only 1% remains of the 2.5% target.
assert abs(coupon_tarn["coupons_paid"][0] - 0.015) < 1e-12
assert abs(coupon_tarn["coupons_paid"][1] - 0.010) < 1e-12
assert abs(sum(coupon_tarn["coupons_paid"]) - 0.025) < 1e-12
assert coupon_tarn["redemption_index"] == 1 and coupon_tarn["redeemed_early"]
coupon_snowball = snowball_coupon_profile(0.03, 0.04, [0.02, 0.03, 0.05], 0, 0.10)
assert all(abs(actual - expected) < 1e-12 for actual, expected in
           zip(coupon_snowball, [0.05, 0.06, 0.05], strict=True))
coupon_inverse = inverse_floater_coupon_profile(0.08, [0.02, 0.03, 0.05], 0, 0.10, 1.5)
assert all(abs(actual - expected) < 1e-12 for actual, expected in
           zip(coupon_inverse, [0.05, 0.035, 0.005], strict=True))
coupon_range = callable_range_accrual_accrued(0.01, 0.03, [0.005, 0.02, 0.03, 0.04], 0.08, 0.25)
assert abs(coupon_range - 0.08 * 0.25 * 2 / 4) < 1e-12
print("TARN payments and target:", coupon_tarn)
print("Snowball coupons:", coupon_snowball)
print("Inverse floater coupons:", coupon_inverse)
print(f"Uncalled range-accrual coupon: {coupon_range:.2%} of notional")


## A real two-asset pool, hand-reconciled

Build the pool from positive collateral records, allocate its one-period cash, and reconcile every discounted tranche receipt.


In [ ]:
import json
from datetime import date
import pandas as pd
from _shared import analyst_tracks as tracks
from finstack_quant.valuations.instruments import structured_credit_tranche_metrics
AS_OF = date(2025, 1, 15)

lesson28_market = tracks.build_structured_market(AS_OF)
lesson28_deal = tracks.clo_deal(one_period=True)
lesson28_spec = lesson28_deal["instrument"]["spec"]
lesson28_spec["pool"]["assets"] = lesson28_spec["pool"]["assets"][:2]
for asset in lesson28_spec["pool"]["assets"]:
    asset["balance"]["amount"] = "50000000"
assert sum(float(a["balance"]["amount"]) for a in lesson28_spec["pool"]["assets"]) == 100_000_000
accrual = (date(2025, 4, 15) - AS_OF).days / 360
cash_available = 100_000_000 * (1 + 0.08 * accrual)
receipts = {"AAA": 80_000_000 * (1 + 0.04 * accrual),
            "BBB": 15_000_000 * (1 + 0.06 * accrual)}
receipts["EQUITY"] = cash_available - sum(receipts.values())
flow_df = lesson28_market.get_discount("USD-OIS").df((date(2025, 4, 15) - AS_OF).days / 365)
waterfall_rows = []
for tranche, receipt in receipts.items():
    native = structured_credit_tranche_metrics(json.dumps(lesson28_deal), tranche, lesson28_market, AS_OF)
    assert abs(native.pv - receipt * flow_df) < 0.01
    waterfall_rows.append({"tranche": tranche, "receipt_usd": receipt, "native_pv_usd": native.pv})
assert abs(sum(r["native_pv_usd"] for r in waterfall_rows) / flow_df - cash_available) < 0.01
print(pd.DataFrame(waterfall_rows).to_string(index=False))


## Analyst lesson 2.8 — CDR is a writedown threshold, not a price target


In [ ]:
from datetime import date
from finstack_quant.valuations.instruments import (
    structured_credit_tranche_breakeven_cdr, structured_credit_tranche_scenario_table,
    structured_credit_tranche_oas, structured_credit_tranche_discount_margin,
)
AS_OF = date(2025, 1, 15)

long_deal = tracks.clo_deal()
long_wire = json.dumps(long_deal)
mezzanine = structured_credit_tranche_metrics(long_wire, "BBB", lesson28_market, AS_OF)
breakeven = structured_credit_tranche_breakeven_cdr(long_wire, "BBB", lesson28_market, AS_OF)
grid = {"cprs": [0.0], "cdrs": [0.0, 0.01, 0.02, 0.05], "severities": [0.6], "recovery_lag": 6}
scenario_table = structured_credit_tranche_scenario_table(long_wire, "BBB", lesson28_market, AS_OF, grid)
scenario_rows = json.loads(scenario_table.to_json())["cells"]
assert scenario_rows[1]["writedown"] == 0 and scenario_rows[2]["writedown"] > 0
assert 0.01 < breakeven < 0.02
assert scenario_rows[-1]["price"] < scenario_rows[0]["price"]
oas = structured_credit_tranche_oas(long_wire, "BBB", mezzanine.price_pct, lesson28_market, AS_OF)
assert abs(oas.oas) < 1e-6
# This BBB contract pays a fixed coupon. A floating-tranche margin is inapplicable.
discount_margin_error = None
try:
    structured_credit_tranche_discount_margin(long_wire, "BBB", lesson28_market, AS_OF, mezzanine.pv)
except ValueError as error:
    discount_margin_error = str(error)
assert discount_margin_error is not None
print("Fixed-coupon discount-margin rejection:", discount_margin_error)
print("Native tranche metrics:", mezzanine.to_json())
print(pd.DataFrame(scenario_rows).to_string(index=False))
print(f"First-writedown CDR: {breakeven:.4%}; model-price OAS: {oas.oas * 10_000:.6f} bp")
